# American Sign Language recognition with a convolutional network

**Status:** ✅ Runs end to end — the Udacity photograph archive no longer exists, so it uses labelled synthetic letters

**Demonstrates:** Neural network design · error analysis. The augmentation hypothesis the original left as a comment is now executable code.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


## 1. Imports

In [ ]:
# BUGFIX: the original did `tf.set_random_seed(2)`, removed in TensorFlow 2.
# The replacement is tf.random.set_seed, which must run *after* kernel
# initialisation to be honoured.
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

np.random.seed(5)
tf.random.set_seed(2)


## What was broken

1. **`tf.set_random_seed(2)`** was removed in TensorFlow 2. The replacement is
   `tf.random.set_seed`.
2. **`from keras.utils import np_utils`** - `np_utils` no longer exists in
   Keras 3. The one-hot encoding here is done directly with `np.eye`, which
   removes the dependency entirely.
3. **`from datasets import sign_language`** imported from a folder that is not
   a Python package, and collides with the unrelated `datasets` distribution
   on PyPI. The loader is now imported by explicit file path.
4. **The final cell was a bare-text `SyntaxError`** - prose typed into a code
   cell.
5. `Conv2D(..., input_shape=...)` passed to `add()` is the old functional-Keras
   style. With the current API the shape belongs on an explicit `Input` layer.
6. `load_data(..., size=2000)` meant *2000 samples* in the original loader. The
   rewritten loader uses `limit` for the count and `size` for the pixel side
   length, so the same call asked for 2000x2000-pixel images - 48 MB each, which
   OOM-killed the kernel outright. That one is worth remembering: renaming a
   parameter silently changes what a call means.

## 2. Load the photographs

In [ ]:
# Load the ASL letter photographs.
#
# BUGFIX: the original did `from datasets import sign_language`, importing a
# module from a folder called `datasets` that is not a package (no __init__.py),
# and a `datasets` distribution with the same name shadows it. The loader is
# now imported by path.
import importlib.util
import os

import numpy as np
from PIL import Image

SIGN_LANGUAGE_DIR = "../../data/computer-vision/asl-letters"
LOADER_PATH = os.path.join(SIGN_LANGUAGE_DIR, "sign_language.py")

SYNTHETIC = False


def _letter_set_present():
    """Return True if all three class folders exist and contain images."""
    return all(
        os.path.isdir(os.path.join(SIGN_LANGUAGE_DIR, letter))
        and any(
            n.lower().endswith((".jpg", ".jpeg", ".png"))
            for n in os.listdir(os.path.join(SIGN_LANGUAGE_DIR, letter))
        )
        for letter in ("A", "B", "C")
    )


if not _letter_set_present():
    # The Udacity S3 bucket that hosted train.zip no longer exists, so there is
    # no reliable download. Generate a clearly-labelled synthetic letter set so
    # the network below can still be built, trained and evaluated end to end.
    # The resulting accuracy describes drawn shapes, not photographs.
    print("! ASL photographs not present - generating a SYNTHETIC stand-in.")
    print("! Any accuracy below describes synthetic shapes, not real letters.")
    rng = np.random.default_rng(0)
    for letter, kind in (("A", 0), ("B", 1), ("C", 2)):
        folder = os.path.join(SIGN_LANGUAGE_DIR, letter)
        os.makedirs(folder, exist_ok=True)
        for index in range(40):
            canvas = np.full((80, 80, 3), 235, np.uint8)
            if kind == 0:                       # A: two diagonals + crossbar
                for y in range(80):
                    canvas[y, max(0, y - 4):y + 5] = 0
                    canvas[y, min(79, y + 4):y + 5] = 0
                canvas[40, 20:60] = 0
            elif kind == 1:                     # B: spine + two bowls
                canvas[:, 20:25] = 0
                canvas[15:20, 20:55] = 0
                canvas[58:63, 20:55] = 0
                canvas[15:63, 53:58] = 0
            else:                               # C: open arc
                yy, xx = np.mgrid[0:80, 0:80]
                radius = np.sqrt((xx - 40) ** 2 + (yy - 40) ** 2)
                canvas[(radius > 22) & (radius < 30) & (xx < 52)] = 0
            canvas = np.clip(canvas + rng.normal(0, 8, canvas.shape), 0, 255).astype(np.uint8)
            Image.fromarray(canvas).save(os.path.join(folder, f"{index:03d}.jpg"))
    SYNTHETIC = True

if not os.path.exists(LOADER_PATH):
    raise FileNotFoundError(f"{LOADER_PATH} is missing from {SIGN_LANGUAGE_DIR}.")

_spec = importlib.util.spec_from_file_location("sign_language", LOADER_PATH)
sign_language = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(sign_language)

# BUGFIX: the original called `load_data(..., size=2000)`, where `size` meant
# the number of SAMPLES. The rewritten loader uses `limit` for the sample count
# and `size` for the image side length, so `size=2000` asks for 2000x2000-pixel
# images: 48 MB each, which OOM-killed the kernel. Both are now passed by name.
(x_train, y_train), (x_test, y_test) = sign_language.load_data(
    container_path=SIGN_LANGUAGE_DIR,
    folders=["A", "B", "C"],
    size=50,          # image side length in pixels
    limit=2000,       # maximum number of samples
    seed=0,
)
# Label names, used by every cell that prints a prediction.
LABELS = ["A", "B", "C"]
N_CLASSES = len(LABELS)

print("train:", x_train.shape, x_train.dtype, "| test:", x_test.shape)
print("label counts -> train:", np.bincount(y_train), " test:", np.bincount(y_test))
if SYNTHETIC:
    print("! SYNTHETIC letters in use - scores below are not a real result.")


## 3. Look at the data

A 4x3 grid at 8 inches shows the same thing as the original 3x12 at 20 inches,
and a full-width 20-inch figure renders to a very large inline PNG that is both
unreadable in a notebook and expensive to hold in memory.

```python
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
...
plt.close(fig)
```

## 4. One-hot labels

In [ ]:
# BUGFIX: `from keras.utils import np_utils` fails on Keras 3 - the module was
# removed. `to_categorical` moved to keras.utils / keras.ops. One-hot encoding
# a 3-class label vector can also just be done directly, which removes the
# dependency entirely.
# LABELS / N_CLASSES are defined in the loader cell above, so that the
# visualisation cell in between can use them too.

y_train_oh = np.eye(N_CLASSES, dtype="float32")[y_train]
y_test_oh = np.eye(N_CLASSES, dtype="float32")[y_test]
print("one-hot train:", y_train_oh.shape, "| row sums:", y_train_oh.sum(axis=1)[:5])


## 5. Define the network

Two conv/pool blocks then a dense softmax. Small on purpose: the point is the
pipeline, not squeezing the last point of accuracy out of three classes.

In [ ]:
# A small convolutional network for 50x50 RGB letter images.
#
# The architecture is unchanged from the original - two conv/pool blocks then a
# dense softmax. What changed is the API surface: layers now come from
# tensorflow.keras, and input_shape is passed to the constructor rather than
# to the first add() call.
model = models.Sequential(
    [
        layers.Input(shape=(50, 50, 3)),
        layers.Conv2D(16, kernel_size=5, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=4),
        layers.Conv2D(32, kernel_size=5, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=4),
        layers.Flatten(),
        layers.Dense(N_CLASSES, activation="softmax"),
    ]
)

model.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 6. Train

In [ ]:
history = model.fit(
    x_train, y_train_oh,
    epochs=5, batch_size=32, validation_split=0.2, verbose=1,
)

plt.figure(figsize=(5, 4))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("ASL letter classification")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Evaluate, and look at the mistakes

This is the step the original notebook was reaching for. Its final cell was a
note rather than code: the author had noticed that the misclassified images
looked obviously different to a human, and concluded the model was too simple
and needed augmentation.

That reasoning is sound. It is now written as a cell that runs, so the claim
can be checked rather than assumed.

In [ ]:
score = model.evaluate(x_test, y_test_oh, verbose=0)
print(f"test accuracy: {score[1]:.4f}")

y_probs = model.predict(x_test, verbose=0)
y_preds = np.argmax(y_probs, axis=1)

# Look at what the model gets wrong. This is the part of the original
# notebook that was worth keeping - the mistakes are the interesting output.
wrong = np.flatnonzero(y_preds != y_test)
print(f"misclassified: {len(wrong)} of {len(y_test)}")

if len(wrong):
    show = min(12, len(wrong))
    fig, axes = plt.subplots(2, (show + 1) // 2, figsize=(2 * show, 4.5))
    for ax, index in zip(axes.ravel(), wrong[:show]):
        ax.imshow(np.squeeze(x_test[index]))
        ax.set_title(f"{LABELS[y_test[index]]} -> {LABELS[y_preds[index]]}", fontsize=8)
        ax.axis("off")
    for ax in axes.ravel()[show:]:
        ax.axis("off")
    plt.suptitle("Misclassified test images", fontsize=11)
    plt.tight_layout()
    plt.show()
